<a href="https://colab.research.google.com/github/piwmat/test/blob/main/EEG_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install MNE-Python to load EEG data
!pip install mne

In [5]:
# Define the maximum number of steps for trajectories
max_trajectory_length = 10000

### Analiza Szeregu Czasowego z Danych California Housing (median_house_value)

### Analiza Szeregu Czasowego z Danych EEG

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import pandas as pd
import mne
from sklearn.preprocessing import StandardScaler

# Load a sample EEG dataset
data_path = mne.datasets.sample.data_path(verbose=False)
raw_fname = data_path / 'MEG' / 'sample' / 'sample_audvis_raw.fif'
raw = mne.io.read_raw_fif(raw_fname, preload=True, verbose=False)

eeg_channels = mne.pick_types(raw.info, meg=False, eeg=True, exclude='bads')
sygnal_eeg_raw = raw.get_data(picks=eeg_channels[0])[0][-max_trajectory_length:]

# 1. Różnicowanie
sygnal_eeg_diff = np.diff(sygnal_eeg_raw)

# 2. Standaryzacja (poprawia zbieżność modeli takich jak ARIMA)
scaler = StandardScaler()
sygnal_eeg = scaler.fit_transform(sygnal_eeg_diff.reshape(-1, 1)).flatten()

cze_probkowania = raw.info['sfreq']
os_czasu = np.arange(len(sygnal_eeg)) / cze_probkowania

print(f"Sygnał zróżnicowany i wystandaryzowany. Nowa długość: {len(sygnal_eeg)}")

KeyboardInterrupt: 

### Podgląd pierwszych próbek sygnału

In [ ]:
print("Pierwsze 10 próbek sygnału EEG:")
print(sygnal_eeg[:10])

### Wizualizacja sygnału EEG

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(os_czasu, sygnal_eeg, lw=1)
plt.title('Szereg Czasowy: Sygnał EEG')
plt.xlabel('Czas (s)')
plt.ylabel('Amplituda EEG (mV)')
plt.grid(True)
plt.show()

### Przygotowanie danych do modelu ARIMA: Podział na zestaw treningowy i testowy

Podzielimy sygnał EEG na 80% danych treningowych i 20% danych testowych. Zestaw treningowy posłuży do nauczenia modelu ARIMA, a testowy do oceny jego przewidywań.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error

# Ponowny podział zróżnicowanego sygnału
rozmiar_treningowy = int(len(sygnal_eeg) * 0.8)
dane_treningowe, dane_testowe = sygnal_eeg[0:rozmiar_treningowy], sygnal_eeg[rozmiar_treningowy:]

print(f"Długość danych treningowych (zróżnicowanych): {len(dane_treningowe)}")
print(f"Długość danych testowych (zróżnicowanych): {len(dane_testowe)}")

### Dopasowanie modelu ARIMA

Teraz dopasujemy model ARIMA do danych treningowych. Wybierzemy tutaj przykładowe parametry (p,d,q) = (5,1,0). W rzeczywistych zastosowaniach należałoby przeprowadzić analizę autokorelacji i częściowej autokorelacji (ACF/PACF) lub użyć funkcji automatycznego doboru parametrów (np. `auto_arima`).

In [ ]:
# Poprawiony model podglądowy: używamy d=0, ponieważ dane_treningowe są już zróżnicowane
model = ARIMA(dane_treningowe, order=(5, 0, 0))
model_fit = model.fit()
print(model_fit.summary())

### Dwustopniowy Model Predykcji: SARIMAX z Korekcją Residuów

Zgodnie z sugestią, wprowadzamy dwustopniowy model predykcji. Pierwszy model (SARIMAX) będzie trenowany na mniejszej części danych treningowych, a jego błędy (residua) na pozostałej części danych treningowych będą korygowane przez drugi model (KNN oparty na Time Delay Embedding).

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

# 1. ARIMA + KNN
# Faza 1: Trening KNN na residuach
rozmiar_treningowy_base = int(len(dane_treningowe) * 0.3)
dane_treningowe_base = dane_treningowe[0:rozmiar_treningowy_base]
dane_walidacyjne_base = dane_treningowe[rozmiar_treningowy_base:]

model_tmp = ARIMA(dane_treningowe_base, order=(2, 0, 0), enforce_stationarity=False).fit()
prognozy_walidacja = model_tmp.predict(start=len(dane_treningowe_base), end=len(dane_treningowe)-1)
residua_sarimax = dane_walidacyjne_base - prognozy_walidacja

X_embedded = create_time_delay_embedding(residua_sarimax, delay=1, dimension=25)
knn_model = KNeighborsRegressor(n_neighbors=1).fit(X_embedded[:-1], X_embedded[1:] - X_embedded[:-1])

# Faza 2: Naprawa desynchronizacji - trenujemy docelowy model na 100% danych treningowych
model_sarimax_final = ARIMA(dane_treningowe, order=(2, 0, 0), enforce_stationarity=False).fit()
arima_forecast_test = model_sarimax_final.get_forecast(steps=len(dane_testowe)).predicted_mean

# Predykcja residuów przez KNN - NAPRAWIONE INDEKSOWANIE [-1]
predicted_residuals_knn = []
current_state = X_embedded[-1].copy()
for i in range(len(dane_testowe)):
    delta = knn_model.predict(current_state.reshape(1, -1))[0]
    current_state = current_state + delta
    predicted_residuals_knn.append(current_state[-1])

final_predictions_combined = arima_forecast_test + np.array(predicted_residuals_knn)
mse_combined = mean_squared_error(dane_testowe, final_predictions_combined)
print(f"Zsynchronizowany MSE (ARIMA+KNN): {mse_combined:.4e}")

### Time Delay Embedding dla Residuów

Aby modelować residua za pomocą algorytmu KNN, przekształcimy je w wektory stanu przy użyciu techniki Time Delay Embedding (osadzanie opóźnień czasowych). Pozwoli to KNN "zobaczyć" kontekst czasowy każdego residuum.

In [ ]:
def create_time_delay_embedding(data, delay, dimension):
    X = []
    for i in range(len(data) - (dimension - 1) * delay):
        state_vector = [data[i + j * delay] for j in range(dimension)]
        X.append(state_vector)
    return np.array(X)

delay = 1
dimension = 25

# Create embedding from the available residuals
X_embedded = create_time_delay_embedding(residua_sarimax, delay, dimension)
print(f"State vectors shape (ARIMA): {X_embedded.shape}")

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# Przygotowanie danych dla modelu KNN
# X_knn to wektory stanu w czasie t
# y_knn to różnica wektora stanu między t+1 a t (zmiana wektora stanu)

X_knn = X_embedded[:-1] # Wszystkie wektory stanu oprócz ostatniego

# Obliczanie różnic wektorów stanu (Wektor_stanu(t+1) - Wektor_stanu(t))
y_knn = X_embedded[1:] - X_embedded[:-1]

print(f"Kształt X_knn: {X_knn.shape}")
print(f"Kształt y_knn (różnice wektorów stanu): {y_knn.shape}")

# Inicjalizacja i trenowanie modelu KNN (k=1)
knn_model = KNeighborsRegressor(n_neighbors=1)
knn_model.fit(X_knn, y_knn)

print("Model KNN dla korekcji residuów został wytrenowany.")

### Predykcja Połączonego Modelu ARIMA z Korekcją KNN

Teraz, gdy oba modele są wytrenowane, połączymy ich predykcje. Model ARIMA dostarczy bazowej prognozy, a model KNN, trenowany na residuach, będzie korygował te prognozy, przewidując przyszłe zachowanie błędów.

In [ ]:
### Sekcja usunięta (Zastąpiona przez zsynchronizowaną wersję w komórce 68378efe)

### Porównanie: Model Liniowy + Korekcja KNN

Teraz zaimplementujemy model liniowy jako bazowy, aby porównać jego wydajność z modelem ARIMA, zwłaszcza w kontekście korekcji residuów za pomocą KNN. Model liniowy nie zakłada konkretnej struktury autokorelacji i jest trenowany metodą najmniejszych kwadratów na opóźnionych wartościach sygnału.

In [ ]:
from sklearn.linear_model import LinearRegression

n_lags = dimension

def create_features_targets(data, n_lags):
    X, y = [], []
    for i in range(n_lags, len(data)):
        X.append(data[i-n_lags:i])
        y.append(data[i])
    return np.array(X), np.array(y)

# Trenowanie na 100% z 8,000 próbek treningowych
X_train_lr, y_train_lr = create_features_targets(dane_treningowe, n_lags)
lr_model_fit = LinearRegression().fit(X_train_lr, y_train_lr)

# Obliczanie residuów na ostatnich 30% z 8,000 próbek (2,400 próbek)
index_70 = int(len(dane_treningowe) * 0.70)
dane_walidacyjne_lr_new = dane_treningowe[index_70:]
full_validation_series_lr = dane_treningowe[index_70 - n_lags:]

X_val_lr_features = np.array([full_validation_series_lr[i-n_lags:i] for i in range(n_lags, len(full_validation_series_lr))])
prognozy_lr_walidacja = lr_model_fit.predict(X_val_lr_features)
residua_lr = dane_walidacyjne_lr_new - prognozy_lr_walidacja

print(f"Ilość obliczonych residuów LR: {len(residua_lr)}")

### Time Delay Embedding dla Residuów Modelu Liniowego

Podobnie jak dla modelu ARIMA, zastosujemy Time Delay Embedding do residuów wygenerowanych przez model liniowy.

In [ ]:
# Using the residuals calculated in the linear model cell
X_embedded_lr = create_time_delay_embedding(residua_lr, delay, dimension)
print(f"State vectors shape (LR): {X_embedded_lr.shape}")

### Trenowanie KNN dla Korekcji Residuów Modelu Liniowego

Wytrenujemy nowy model KNN na residuach z modelu liniowego.

In [ ]:
# Przygotowanie danych dla modelu KNN (z residuów LR)
X_knn_lr = X_embedded_lr[:-1]
y_knn_lr = X_embedded_lr[1:] - X_embedded_lr[:-1]

# Inicjalizacja i trenowanie modelu KNN (k=1) dla residuów LR
knn_model_lr = KNeighborsRegressor(n_neighbors=1)
knn_model_lr.fit(X_knn_lr, y_knn_lr)

print("Model KNN dla korekcji residuów modelu liniowego został wytrenowany.")

### Predykcja Połączonego Modelu Liniowego z Korekcją KNN

Teraz połączymy prognozy modelu liniowego z korekcją residuów KNN i ocenimy jego wydajność.

In [ ]:
from sklearn.linear_model import LinearRegression

# 2. LR + KNN
X_train_lr, y_train_lr = create_features_targets(dane_treningowe, n_lags=25)
lr_model_fit = LinearRegression().fit(X_train_lr, y_train_lr)

index_70 = int(len(dane_treningowe) * 0.70)
full_validation_series_lr = dane_treningowe[index_70 - 25:]
X_val_lr_features = np.array([full_validation_series_lr[i-25:i] for i in range(25, len(full_validation_series_lr))])
residua_lr = dane_treningowe[index_70:] - lr_model_fit.predict(X_val_lr_features)

X_embedded_lr = create_time_delay_embedding(residua_lr, delay=1, dimension=25)
knn_model_lr = KNeighborsRegressor(n_neighbors=1).fit(X_embedded_lr[:-1], X_embedded_lr[1:] - X_embedded_lr[:-1])

# Predykcja
lr_forecast_test = []
current_window = dane_treningowe[-25:].copy()
for i in range(len(dane_testowe)):
    next_p = lr_model_fit.predict(current_window.reshape(1, -1))[0]
    lr_forecast_test.append(next_p)
    current_window = np.append(current_window[1:], next_p)

# Predykcja residuów przez KNN - NAPRAWIONE INDEKSOWANIE [-1]
predicted_residuals_knn_lr = []
curr_res_state = X_embedded_lr[-1].copy()
for i in range(len(dane_testowe)):
    delta = knn_model_lr.predict(curr_res_state.reshape(1, -1))[0]
    curr_res_state = curr_res_state + delta
    predicted_residuals_knn_lr.append(curr_res_state[-1])

final_predictions_combined_lr = np.array(lr_forecast_test) + np.array(predicted_residuals_knn_lr)
mse_combined_lr = mean_squared_error(dane_testowe, final_predictions_combined_lr)
print(f"Nowy MSE (LR+KNN): {mse_combined_lr:.4e}")

In [4]:
from IPython.display import Markdown

# Retrieve current state parameters
try:
    intercept_lr = lr_model_fit.intercept_
    coefficients_lr = lr_model_fit.coef_
    coef_str = ', '.join([f'{c:.2e}' for c in coefficients_lr[:5]]) + " ..."
    mse_val = f"{mse_combined_lr:.4e}"
    p_val = n_lags
except NameError:
    intercept_lr = 0.0
    coef_str = "Model nie został jeszcze wytrenowany"
    mse_val = "N/A"
    p_val = 25

latex_formula_str = r"$$y_{t+1} = \beta_0 + \beta_1 y_t + \beta_2 y_{t-1} + \dots + \beta_p y_{t-p+1}$$"

markdown_content = f"""
### Podsumowanie: Wytrenowany Model Liniowy + KNN

Projekt został dostosowany do rzeczywistych danych EEG oraz ograniczenia trajektorii do ostatnich {max_trajectory_length} kroków.

#### Model Liniowy (Bazowy)
Model przewiduje wartość `y` w czasie `t+1` przy użyciu `p={p_val}` opóźnień:

{latex_formula_str}

**Parametry modelu:**
- **Wyraz wolny (intercept)** $\beta_0$: `{intercept_lr:.2e}`
- **Pierwsze 5 współczynników ($\beta_1 \dots \beta_5$):** `{coef_str}`

#### Wyniki Końcowe
- **Zbiór danych:** Realne sygnały EEG (kanał EGG)
- **Ograniczenie trajektorii:** {max_trajectory_length} kroków
- **Mean Squared Error (MSE) Liniowy+KNN:** `{mse_val}`
"""

display(Markdown(markdown_content))

<>:37: SyntaxWarning: invalid escape sequence '\d'
<>:37: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_2816/3165305428.py:37: SyntaxWarning: invalid escape sequence '\d'
  



### Podsumowanie: Wytrenowany Model Liniowy + KNN

Projekt został dostosowany do rzeczywistych danych EEG oraz ograniczenia trajektorii do ostatnich 10000 kroków.

#### Model Liniowy (Bazowy)
Model przewiduje wartość `y` w czasie `t+1` przy użyciu `p=25` opóźnień:

$$y_{t+1} = \beta_0 + \beta_1 y_t + \beta_2 y_{t-1} + \dots + \beta_p y_{t-p+1}$$

**Parametry modelu:**
- **Wyraz wolny (intercept)** $eta_0$: `0.00e+00`
- **Pierwsze 5 współczynników ($eta_1 \dots eta_5$):** `Model nie został jeszcze wytrenowany`

#### Wyniki Końcowe
- **Zbiór danych:** Realne sygnały EEG (kanał EGG)
- **Ograniczenie trajektorii:** 10000 kroków
- **Mean Squared Error (MSE) Liniowy+KNN:** `N/A`


### Model 3: Exponential Smoothing (ETS) + Korekcja KNN

Trzecim modelem bazowym będzie Wygładzanie Wykładnicze (Exponential Smoothing). Model ten jest szczególnie skuteczny w wykrywaniu trendów i sezonowości w szeregach czasowych.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# 3. ETS + KNN
model_ets = ExponentialSmoothing(dane_treningowe, trend=None, seasonal=None).fit()
residua_ets = dane_treningowe - model_ets.fittedvalues
X_embedded_ets = create_time_delay_embedding(residua_ets, delay=1, dimension=25)
knn_model_ets = KNeighborsRegressor(n_neighbors=1).fit(X_embedded_ets[:-1], X_embedded_ets[1:] - X_embedded_ets[:-1])

ets_forecast_test = model_ets.forecast(len(dane_testowe))

# Predykcja residuów przez KNN - NAPRAWIONE INDEKSOWANIE [-1]
predicted_residuals_knn_ets = []
curr_res_state_ets = X_embedded_ets[-1].copy()
for i in range(len(dane_testowe)):
    delta = knn_model_ets.predict(curr_res_state_ets.reshape(1, -1))[0]
    curr_res_state_ets = curr_res_state_ets + delta
    predicted_residuals_knn_ets.append(curr_res_state_ets[-1])

final_predictions_combined_ets = ets_forecast_test + np.array(predicted_residuals_knn_ets)
mse_combined_ets = mean_squared_error(dane_testowe, final_predictions_combined_ets)
print(f"Nowy MSE (ETS+KNN): {mse_combined_ets:.4e}")

In [ ]:
# 6. Wizualizacja wyników dla ETS + KNN (używając poprawnych zmiennych z komórki 768d2fd7)
plt.figure(figsize=(15, 6))
plt.plot(os_czasu_test, dane_testowe, label='Rzeczywiste EEG', color='orange', alpha=0.7)
plt.plot(os_czasu_test, final_predictions_combined_ets, label='ETS + KNN (Poprawiony)', color='cyan', linestyle='--')
plt.title(f'Sygnał EEG: Model ETS (bez trendu) + KNN')
plt.legend()
plt.grid(True)
plt.show()

### Porównanie wszystkich modeli

Poniżej znajduje się zestawienie błędów średniokwadratowych (MSE) dla wszystkich trzech przetestowanych podejść: ARIMA, Regresja Liniowa oraz Exponential Smoothing (ETS) - każdy z korekcją KNN.

In [ ]:
# Usunięto redundantną tabelę pośrednią

### Porównanie wszystkich modeli

Poniżej znajduje się zestawienie błędów średniokwadratowych (MSE) dla wszystkich trzech przetestowanych podejść.

In [ ]:
# Usunięto redundantną tabelę pośrednią

### Kompleksowa Wizualizacja: Podział na Trening/Test i Predykcja Trajektorii

Poniższy wykres przedstawia cały analizowany wycinek sygnału (10,000 próbek) z podziałem na część, na której model się uczył, oraz część testową wraz z nałożoną predykcją hybrydową (Linear Regression + KNN).

In [ ]:
import matplotlib.pyplot as plt

os_czasu_trening = os_czasu[:rozmiar_treningowy]
os_czasu_test = os_czasu[rozmiar_treningowy:]

plt.figure(figsize=(16, 8))

plt.plot(os_czasu_trening, dane_treningowe, label='Dane treningowe (Historyczne)', color='gray', alpha=0.4)
plt.plot(os_czasu_test, dane_testowe, label='Dane testowe (Rzeczywiste)', color='orange', alpha=0.8, lw=2)

plt.plot(os_czasu_test, final_predictions_combined, label='ARIMA + KNN', color='blue', linestyle='--', alpha=0.6)
plt.plot(os_czasu_test, final_predictions_combined_lr, label='LR + KNN', color='red', linestyle='-', alpha=0.6)
plt.plot(os_czasu_test, final_predictions_combined_ets, label='ETS + KNN', color='cyan', linestyle=':', alpha=0.6)
plt.plot(os_czasu_test, ensemble_predictions, label='Ensemble (Mean)', color='black', linestyle='-', lw=2, alpha=0.9)

plt.axvline(x=os_czasu_trening[-1], color='black', linestyle='-', alpha=0.5, label='Granica Trening/Test')

plt.title(f'Kompleksowa analiza wszystkich modeli (w tym Ensemble) na trajektorii EEG')
plt.xlabel('Czas (s)')
plt.ylabel('Amplituda')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### Zbliżenie: Trajektoria Testowa vs Predykcja

Ta sekcja skupia się wyłącznie na fragmencie testowym (2000 próbek), aby umożliwić wizualną ocenę precyzji modelu hybrydowego.

In [ ]:
plt.figure(figsize=(15, 7))

plt.plot(os_czasu_test, dane_testowe, label="Rzeczywisty sygnał (Test)", color="orange", lw=2, alpha=0.5)
plt.plot(os_czasu_test, final_predictions_combined, label="ARIMA + KNN", color="blue", linestyle="--", alpha=0.5)
plt.plot(os_czasu_test, final_predictions_combined_lr, label="LR + KNN", color="red", linestyle="-", alpha=0.5)
plt.plot(os_czasu_test, final_predictions_combined_ets, label="ETS + KNN", color="cyan", linestyle=":", alpha=0.5)
plt.plot(os_czasu_test, ensemble_predictions, label="Ensemble (Mean)", color="black", lw=2.5)

plt.title("Ostateczne porównanie: Wszystkie modele hybrydowe vs Ensemble")
plt.xlabel("Czas (s)")
plt.ylabel("Amplituda")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Model 4: Ensemble (Średnia z modeli hybrydowych)

Ten model łączy predykcje ARIMA+KNN, LR+KNN oraz ETS+KNN, obliczając ich średnią arytmetyczną dla każdego punktu w czasie.

In [ ]:
# 4. Ensemble (Naprawiony przez synchronizację ARIMA)
ensemble_predictions = (final_predictions_combined + final_predictions_combined_lr + final_predictions_combined_ets) / 3
mse_ensemble = mean_squared_error(dane_testowe, ensemble_predictions)

print(f"Poprawny MSE (Ensemble): {mse_ensemble:.4e}")

results_data = {
    'Model': ['ARIMA+KNN', 'LR+KNN', 'ETS+KNN', 'Ensemble'],
    'MSE': [mse_combined, mse_combined_lr, mse_combined_ets, mse_ensemble]
}
display(pd.DataFrame(results_data).sort_values('MSE'))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Pobieramy oryginalny sygnał (przed różnicowaniem) dla celów porównawczych i punktu startowego
raw_eeg_full = raw.get_data(picks=eeg_channels[0])[0][-max_trajectory_length:]

# Punkt podziału w oryginalnej skali
# Sygnał różnicowany (sygnal_eeg) ma długość N-1 względem raw_eeg_full (N)
last_train_val = raw_eeg_full[rozmiar_treningowy]

def inverse_diff(predictions, last_value):
    """Odwraca różnicowanie (kumulatywna suma startująca od last_value)"""
    return last_value + np.cumsum(predictions)

# 2. Odwracanie różnicowania dla wszystkich modeli i danych testowych
dane_testowe_original = raw_eeg_full[rozmiar_treningowy+1:]

# Uwaga: sygnal_eeg[t] = raw[t+1] - raw[t].
# Predykcje testowe odpowiadają różnicom dla indeksów od rozmiar_treningowy+1 wzwyż.
preds_arima_orig = inverse_diff(final_predictions_combined, last_train_val)
preds_lr_orig = inverse_diff(final_predictions_combined_lr, last_train_val)
preds_ets_orig = inverse_diff(final_predictions_combined_ets, last_train_val)
preds_ensemble_orig = inverse_diff(ensemble_predictions, last_train_val)

# 3. Wizualizacja na nieróżnicowanych danych (zbliżenie na test)
os_czasu_test_orig = os_czasu[rozmiar_treningowy:]

plt.figure(figsize=(15, 7))
plt.plot(os_czasu_test_orig, dane_testowe_original, label="Rzeczywisty sygnał EEG (Oryginalny)", color="orange", lw=2, alpha=0.6)
plt.plot(os_czasu_test_orig, preds_arima_orig, label="ARIMA + KNN (Oryg. skala)", color="blue", linestyle="--", alpha=0.7)
plt.plot(os_czasu_test_orig, preds_lr_orig, label="LR + KNN (Oryg. skala)", color="red", linestyle="-", alpha=0.7)
plt.plot(os_czasu_test_orig, preds_ets_orig, label="ETS + KNN (Oryg. skala)", color="cyan", linestyle=":", alpha=0.7)
plt.plot(os_czasu_test_orig, preds_ensemble_orig, label="Ensemble (Mean - Oryg. skala)", color="black", lw=2.5)

plt.title("Prognoza na oryginalnej skali sygnału EEG (po odwróceniu różnicowania)")
plt.xlabel("Czas (s)")
plt.ylabel("Amplituda (V)")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Wyliczenie MSE na oryginalnej skali
mse_orig_ensemble = mean_squared_error(dane_testowe_original, preds_ensemble_orig)
print(f"MSE Ensemble na oryginalnej skali: {mse_orig_ensemble:.4e}")

In [ ]:
# Zbiorcze zestawienie wszystkich wyników na końcu projektu
results_data = {
    'Model': ['ARIMA + KNN', 'LR + KNN', 'ETS + KNN', 'Ensemble'],
    'MSE': [mse_combined, mse_combined_lr, mse_combined_ets, mse_ensemble]
}

df_results_final = pd.DataFrame(results_data).drop_duplicates(subset=['Model'])
print("Ostateczny ranking modeli (Skala zróżnicowana i wystandaryzowana):")
display(df_results_final.sort_values(by='MSE'))

In [ ]:
plt.figure(figsize=(15, 7))

plt.plot(os_czasu_test, dane_testowe, label="Rzeczywisty sygnał", color="orange", lw=2, alpha=0.4)
plt.plot(os_czasu_test, ensemble_predictions, label="Ensemble (Mean)", color="black", linestyle="-", lw=2)

plt.title("Predykcja modelu Ensemble na tle sygnału rzeczywistego")
plt.xlabel("Czas (s)")
plt.ylabel("Amplituda")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Sekcja usunięta (Zbiór Iris nie dotyczy analizy EEG)

In [ ]:
# Komórka usunięta - brak powiązania z projektem